# معمل التجارب: التقطيع الدلالي الجاهز باستخدام LangChain (المحاضرة 40)
## القسم الأول: نظرة عامة وتوجيهات المعمل

في هذا المعمل العملي:
1. ستتعلم وتطبق استخدام أداة `SemanticChunker` الرسمية المدمجة في LangChain.
2. ستختبر استراتيجيات العتبات الإحصائية المختلفة (`percentile`, `standard_deviation`, `interquartile`).
3. ستطبق تكليف المحاضرة ببناء مسار RAG كامل باستخدام `FAISS` وموديل `Groq LLM` وسلسلة `LCEL`.


## القسم الثاني: الكود المرجعي للتطبيق والتجربة
شغل الخلايا التالية لمتابعة عمل الخوارزمية، ثم استخدم مساحة التجربة الحرة في الأسفل للتعديل.

In [ ]:
# 1. استدعاء المكتبات وإعداد المستند
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

loader = TextLoader("../../stages/04-advanced-chunking/04-semantic-chunking-langchain/langchain_intro.txt")
docs = loader.load()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"[INFO] Document loaded. Characters: {len(docs[0].page_content)}")

In [ ]:
# 2. تشغيل SemanticChunker الافتراضي
chunker = SemanticChunker(embeddings)
chunks = chunker.split_documents(docs)

print(f"[INFO] Total chunks produced: {len(chunks)}\n")
for idx, c in enumerate(chunks, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{c.page_content}\n")

In [ ]:
# 3. تكليف المحاضرة: بناء مسار RAG كامل مع FAISS و Groq
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_template("""Answer based only on the context:
Context: {context}
Question: {question}
Answer:""")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke({"question": "What tools can LangChain be combined with?"})
print(f"[RESPONSE]:\n{response}")

---
## القسم الثالث: مساحة التجربة والتطبيق العملي
استخدم الخلية التالية لتجربة نصوص جديدة، أو اختبار أنواع عتبات مختلفة (مثل standard_deviation أو interquartile)، أو طرح أسئلة جديدة.

In [ ]:
# مساحتك للتجربة والتطبيق العملي:
# جرب استدعاء SemanticChunker بنوع عتبة مخصص أو اختبر سؤالاً جديداً هنا:

